# Game 2 - Optimized 1-vs-1 Pairs Trading (Coffee, G17)

## 1. Imports

In [1]:
import os

try:
    from google.colab import files
    print("Google Colab detecte — upload le fichier coffee_dataset_G17.csv")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
except ImportError:
    CSV_PATH = 'coffee_dataset_G17.csv'
    print(f"Execution locale — fichier : {CSV_PATH}")

os.makedirs('figures', exist_ok=True)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import adfuller

pd.set_option('display.float_format', '{:.4f}'.format)
print('imports OK')

Execution locale — fichier : coffee_dataset_G17.csv


imports OK


## 2. Chargement & Nettoyage

In [2]:
df_raw=pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
df_raw.index.name='Date'
df_raw=df_raw.sort_index()

df_clean=df_raw.dropna(thresh=int(0.80 * len(df_raw)), axis=1).ffill().dropna()
for col in df_clean.columns:
    ret=df_clean[col].pct_change().dropna()
    bad=df_clean.index[1:][np.abs((ret - ret.mean()) / ret.std()) > 5]
    df_clean.loc[bad, col]=np.nan
df_clean=df_clean.ffill().dropna()

ANCHOR='KC=F'
STOCKS=[c for c in df_clean.columns if c != ANCHOR]
print(f'{df_clean.shape}  |  {df_clean.index[0].date()} -> {df_clean.index[-1].date()}')
df_clean.head()

(1093, 10)  |  2021-10-20 -> 2026-02-27


,BROS,DBA,FARM,KC=F,KDP,MDLZ,NSRGY,SBUX,SJM,WEST
Date,,,,,,,,,,
2021-10-20,61.4200,16.9252,7.9900,205.5500,31.2751,54.0325,112.5147,102.3129,104.4686,9.6900
2021-10-21,60.5800,16.7575,7.7300,203.3000,30.8091,53.9159,114.0060,102.8340,104.5029,9.6900
2021-10-22,68.8600,16.7487,7.3900,199.8500,31.0600,54.2386,114.3453,102.9329,105.5067,9.6900
2021-10-25,66.9400,16.8899,7.4300,202.5500,30.8628,54.0773,113.2737,103.0766,105.6526,9.7000
2021-10-26,70.4100,16.9869,7.6300,208.1000,30.9883,54.5164,113.8809,103.2653,106.9910,9.7000


## 3. Visualisation interactive des prix et spreads

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

log_p=np.log(df_clean)
norm=(df_clean / df_clean.iloc[0]) * 100
stocks_list=[c for c in df_clean.columns if c != ANCHOR]
t_split=int(0.75 * len(df_clean))
split_date=str(df_clean.index[t_split].date())

colors=['blue','orange','green','red','purple','brown','pink','gray','olive','cyan','black']

fig=make_subplots(
    rows=2, cols=1,
    subplot_titles=('Prix normalises base 100', 'Spread brut:log(KC=F) - log(stock)'),
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.5, 0.5]
)

for i, col in enumerate(df_clean.columns):
    c=colors[i % len(colors)]
    lw=2.5 if col == ANCHOR else 1.2
    fig.add_trace(go.Scatter(
        x=norm.index, y=norm[col], name=col,
        line=dict(color=c, width=lw),
        hovertemplate=f'<b>{col}</b><br>%{{x|%Y-%m-%d}}<br>Base 100:%{{y:.1f}}<extra></extra>'
    ), row=1, col=1)

for i, stock in enumerate(stocks_list):
    spread=log_p[ANCHOR] - log_p[stock]
    c=colors[(i+1) % len(colors)]
    fig.add_trace(go.Scatter(
        x=spread.index, y=spread, name=f'KC=F - {stock}',
        line=dict(color=c, width=1.0),
        hovertemplate=f'<b>KC=F - {stock}</b><br>%{{x|%Y-%m-%d}}<br>Spread:%{{y:.4f}}<extra></extra>'
    ), row=2, col=1)

for row in [1, 2]:
    fig.add_shape(type='line', x0=split_date, x1=split_date, y0=0, y1=1,
                  xref='x', yref='paper', line=dict(color='black', width=1.5, dash='dash'), row=row, col=1)

fig.add_annotation(x=split_date, y=1.02, xref='x', yref='paper',
                   text='IS / OOS', showarrow=False, font=dict(size=11))

fig.update_layout(
    height=700,
    title_text='Coffee G17 — Vue globale des actifs et spreads',
    hovermode='x unified',
    legend=dict(orientation='v', x=1.01, y=1),
    template='plotly_white'
)
fig.update_yaxes(title_text='Base 100', row=1, col=1)
fig.update_yaxes(title_text='log(KC=F) - log(stock)', row=2, col=1)
fig.show()

# version matplotlib statique — sauvegarde pour le rapport
fig_p, ax_p=plt.subplots(figsize=(12, 5))
for i, col in enumerate(df_clean.columns):
    lw=2.0 if col == ANCHOR else 0.9
    alpha=1.0 if col == ANCHOR else 0.6
    ax_p.plot(norm.index, norm[col], label=col, linewidth=lw, alpha=alpha, color=colors[i % len(colors)])
ax_p.axvline(pd.Timestamp(split_date), color='black', linewidth=1.3, linestyle='--', label='IS / OOS')
ax_p.set_title('Prix normalises base 100 de tous les actifs')
ax_p.set_ylabel('Indice base 100')
ax_p.legend(ncol=4, fontsize=8)
ax_p.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig('figures/fig_prices.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Test de Cointegration (Engle-Granger)

In [4]:
log_prices=np.log(df_clean)

N=len(df_clean)
T_train=int(0.75 * N)
train_idx=df_clean.index[:T_train]
test_idx=df_clean.index[T_train:]
log_IS=log_prices.loc[train_idx]

print(f'IS:{train_idx[0].date()} -> {train_idx[-1].date()} ({len(train_idx)} obs)')
print(f'OOS:{test_idx[0].date()}  -> {test_idx[-1].date()} ({len(test_idx)} obs)')

results=[]
for stock in STOCKS:
    y=log_IS[ANCHOR].values
    X=np.column_stack([np.ones(len(log_IS)), log_IS[stock].values])
    beta=np.linalg.lstsq(X, y, rcond=None)[0]
    _, pval, *_=adfuller(y - X @ beta, regression='c')
    results.append({'Asset':stock, 'EG p-value':round(pval, 4), 'Gamma':round(beta[1], 4)})

coint_df=pd.DataFrame(results).sort_values('EG p-value')
print(coint_df.to_string(index=False))

PAIR=coint_df.iloc[0]['Asset']
X_IS=np.column_stack([np.ones(len(log_IS)), log_IS[PAIR].values])
GAMMA=np.linalg.lstsq(X_IS, log_IS[ANCHOR].values, rcond=None)[0][1]
print(f'\nPaire retenue:KC=F / {PAIR}  |  Gamma={GAMMA:.4f}')

IS:2021-10-20 -> 2025-01-24 (819 obs)
OOS:2025-01-27  -> 2026-02-27 (274 obs)


Asset  EG p-value   Gamma
 BROS      0.0163  0.4703
 WEST      0.1560 -0.7099
NSRGY      0.2650 -0.9864
  SJM      0.2997 -1.0005
  DBA      0.5448  0.9404
 MDLZ      0.7266 -0.5923
 SBUX      0.8576 -0.3314
  KDP      0.8674  0.8960
 FARM      0.8708 -0.0359

Paire retenue:KC=F / BROS  |  Gamma=0.4703


## 5. Comparaison des Z-scores par paire

In [5]:
Z_THRESHOLD=0.7
LB_ZSCORE=60

fig=go.Figure()
stds={}
for stock in STOCKS:
    X_tmp=np.column_stack([np.ones(len(log_IS)), log_IS[stock].values])
    g_tmp=np.linalg.lstsq(X_tmp, log_IS[ANCHOR].values, rcond=None)[0][1]
    sp_tmp=log_prices[ANCHOR] - g_tmp * log_prices[stock]
    zs_tmp=((sp_tmp - sp_tmp.rolling(LB_ZSCORE).mean()) / sp_tmp.rolling(LB_ZSCORE).std()).dropna()
    stds[stock]=round(zs_tmp.std(), 4)
    fig.add_trace(go.Scatter(
        x=zs_tmp.index, y=zs_tmp, name=f'KC=F / {stock}',
        line=dict(width=1.2),
        visible=True if stock == PAIR else 'legendonly',
        hovertemplate=f'<b>KC=F / {stock}</b><br>%{{x|%Y-%m-%d}}<br>Z=%{{y:.2f}}<extra></extra>'
    ))

fig.add_hline(y=Z_THRESHOLD, line_dash='dash', line_color='red',   line_width=1, annotation_text=f'+{Z_THRESHOLD}')
fig.add_hline(y=-Z_THRESHOLD, line_dash='dash', line_color='green', line_width=1, annotation_text=f'-{Z_THRESHOLD}')
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.add_shape(type='line', x0=split_date, x1=split_date, y0=0, y1=1,
              xref='x', yref='paper', line=dict(color='black', width=1.5, dash='dot'))
fig.add_annotation(x=split_date, y=1.03, xref='x', yref='paper',
                   text='IS / OOS', showarrow=False, font=dict(size=11))
fig.update_layout(height=450, title=f'Z-score rolling {LB_ZSCORE}j — toutes les paires vs KC=F',
                  hovermode='x unified', template='plotly_white', yaxis_title='Z-score',
                  legend=dict(orientation='v', x=1.01, y=1))
fig.show()

colors_lines=['blue','orange','green','red','purple','brown','pink','gray','olive']
fig_static, ax=plt.subplots(figsize=(12, 5))
zs_all={}
for i, stock in enumerate(STOCKS):
    X_tmp=np.column_stack([np.ones(len(log_IS)), log_IS[stock].values])
    g_tmp=np.linalg.lstsq(X_tmp, log_IS[ANCHOR].values, rcond=None)[0][1]
    sp_tmp=log_prices[ANCHOR] - g_tmp * log_prices[stock]
    zs_tmp=((sp_tmp - sp_tmp.rolling(LB_ZSCORE).mean()) / sp_tmp.rolling(LB_ZSCORE).std()).dropna()
    lw=1.8 if stock == PAIR else 0.8
    alpha=1.0 if stock == PAIR else 0.4
    ax.plot(zs_tmp.index, zs_tmp, label=f'KC=F / {stock}', linewidth=lw, alpha=alpha, color=colors_lines[i % len(colors_lines)])
ax.axhline(Z_THRESHOLD,  color='red',   linestyle='--', linewidth=1, label=f'+{Z_THRESHOLD}')
ax.axhline(-Z_THRESHOLD, color='green', linestyle='--', linewidth=1, label=f'-{Z_THRESHOLD}')
ax.axhline(0, color='black', linewidth=0.7)
ax.axvline(pd.Timestamp(split_date), color='black', linewidth=1.3, linestyle=':', label='IS / OOS')
ax.set_title(f'Z-score rolling {LB_ZSCORE}j � toutes les paires vs KC=F')
ax.set_ylabel('Z-score')
ax.legend(ncol=3, fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig('figures/fig_zscore_lines.png', dpi=150, bbox_inches='tight')
plt.show()


std_df=pd.DataFrame.from_dict(stds, orient='index', columns=['Std Z-score']).sort_values('Std Z-score')
colors_hist=['red' if s == PAIR else 'steelblue' for s in std_df.index]
plt.figure(figsize=(9, 4))
plt.bar(std_df.index, std_df['Std Z-score'], color=colors_hist)
plt.axhline(std_df['Std Z-score'].mean(), color='grey', linestyle='--', linewidth=1, label='Moyenne')
plt.ylabel('Std du Z-score')
plt.title(f'Std du Z-score par paire (lookback={LB_ZSCORE}j) — rouge=paire retenue')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig_zscore_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print(std_df.to_string())

       Std Z-score
WEST        1.2812
BROS        1.3149
SJM         1.3507
NSRGY       1.3631
FARM        1.3704
MDLZ        1.3775
SBUX        1.3858
KDP         1.3923
DBA         1.4461


## 6. Grid Search IS — Lookback optimal

In [6]:
Z_THRESHOLD=0.7
TC=0.001
RF_DAILY=0.04 / 252
spread_IS=log_IS[ANCHOR] - GAMMA * log_IS[PAIR]

def sharpe_is(lookback):
    zs=((spread_IS - spread_IS.rolling(lookback).mean()) / spread_IS.rolling(lookback).std()).dropna()
    pos=pd.Series(0, index=zs.index)
    p=0
    for i, z in enumerate(zs):
        if p == 0:
            if z >  Z_THRESHOLD:p=-1
            elif z < -Z_THRESHOLD:p=1
        elif (p == 1 and z >= 0) or (p == -1 and z <= 0):
            p=0
        pos.iloc[i]=p
    r=(pos.shift(1) * spread_IS.loc[zs.index].diff() - TC * (pos.diff().abs() > 0)).dropna()
    return -999 if r.std() == 0 else (r.mean() - RF_DAILY) / r.std() * np.sqrt(252)

lookback_grid=list(range(5, 361, 5))
sharpes_grid=[sharpe_is(L) for L in lookback_grid]
BEST_LB=lookback_grid[int(np.argmax(sharpes_grid))]
print(f'L*={BEST_LB} jours  |  IS Sharpe={max(sharpes_grid):.3f}')

plt.figure(figsize=(10, 4))
plt.plot(lookback_grid, sharpes_grid, linewidth=1.5)
plt.axvline(BEST_LB, color='red', linestyle='--', label=f'L*={BEST_LB}')
plt.xlabel('Lookback (jours)')
plt.ylabel('Sharpe IS')
plt.title(f'Grid Search — Lookback vs Sharpe IS  (L*={BEST_LB}j)')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig_gridsearch.png', dpi=150, bbox_inches='tight')
plt.show()

L*=145 jours  |  IS Sharpe=1.173


## 7. Backtest OOS

In [7]:
CAPITAL=100_000
VOL_MULT=1.5
VOL_LOOKBACK=30
STOP_BASE=0.025
MIN_HOLD=5

spread_full=log_prices[ANCHOR] - GAMMA * log_prices[PAIR]
zscore=((spread_full - spread_full.rolling(BEST_LB).mean()) / spread_full.rolling(BEST_LB).std()).dropna()
vol_IS_avg=spread_full.loc[train_idx].rolling(VOL_LOOKBACK).std().mean()

zs_OOS=zscore.loc[test_idx]
spread_OOS=spread_full.loc[test_idx]
vol_OOS=spread_OOS.rolling(VOL_LOOKBACK).std()

equity=CAPITAL; position=0; hold_days=0; peak_pnl=0.0
entry_spread=0.0; open_date=None
eq_list=[]; pos_list=[]; trade_log=[]; n_stops=0

for i, date in enumerate(zs_OOS.index):
    z=zs_OOS.iloc[i]
    sp=spread_OOS.iloc[i]
    vol=vol_OOS.iloc[i] if not np.isnan(vol_OOS.iloc[i]) else vol_IS_avg
    stop_adaptive=STOP_BASE * (vol / vol_IS_avg if vol_IS_avg > 0 else 1.0)

    if position != 0:
        hold_days += 1
        trade_pnl=position * (sp - entry_spread)
        if trade_pnl > peak_pnl:peak_pnl=trade_pnl
        exit_z=(position == 1 and z >= 0) or (position == -1 and z <= 0)
        exit_stop=(hold_days >= MIN_HOLD) and (trade_pnl < peak_pnl - stop_adaptive)
        if exit_z or exit_stop:
            equity += (trade_pnl - 2 * TC) * equity
            trade_log.append({'open_date':open_date, 'close_date':date,
                'direction':'LONG' if position == 1 else 'SHORT',
                'net_ret':round(trade_pnl - 2 * TC, 5),
                'stop':exit_stop and not exit_z, 'hold_days':hold_days})
            if exit_stop and not exit_z:n_stops += 1
            position=0; hold_days=0; peak_pnl=0.0

    if position == 0 and vol <= VOL_MULT * vol_IS_avg:
        if z >  Z_THRESHOLD:position=-1; entry_spread=sp; open_date=date; equity -= TC * equity
        elif z < -Z_THRESHOLD:position=1; entry_spread=sp; open_date=date; equity -= TC * equity

    eq_list.append(equity); pos_list.append(position)

portfolio_OOS=pd.DataFrame({'zscore':zs_OOS.values, 'position':pos_list, 'equity':eq_list}, index=zs_OOS.index)
trades=pd.DataFrame(trade_log)
print(f'Trades:{len(trades)}  |  Stops:{n_stops}  |  Gagnants:{(trades["net_ret"] > 0).sum() if len(trades) else 0}')
if len(trades):print(trades.to_string(index=False))

Trades:34  |  Stops:29  |  Gagnants:15
 open_date close_date direction  net_ret  stop  hold_days
2025-01-27 2025-02-03     SHORT  -0.0584  True          5
2025-02-03 2025-02-10     SHORT  -0.0974  True          5
2025-02-10 2025-02-21     SHORT   0.1310  True          8
2025-02-21 2025-02-28     SHORT   0.0703 False          5
2025-03-04 2025-03-11     SHORT  -0.0780  True          5
2025-03-11 2025-03-28     SHORT   0.0661  True         13
2025-03-28 2025-04-04     SHORT  -0.0561  True          5
2025-04-04 2025-04-09     SHORT   0.1420 False          3
2025-04-11 2025-04-21     SHORT  -0.0319  True          5
2025-04-21 2025-04-28     SHORT  -0.1055  True          5
2025-04-28 2025-05-06     SHORT   0.0328  True          6
2025-05-06 2025-05-14     SHORT   0.1565 False          6
2025-05-28 2025-06-16      LONG  -0.0258  True         13
2025-06-16 2025-06-24      LONG  -0.0860  True          5
2025-06-24 2025-07-01      LONG  -0.0332  True          5
2025-07-01 2025-07-09      LONG  

## 8. Metriques

In [8]:
def compute_metrics(eq, label='Strategy'):
    r=eq.pct_change().dropna()
    tr=(eq.iloc[-1] / eq.iloc[0]) - 1
    ar=(1 + tr) ** (252 / len(eq)) - 1
    sh=(r.mean() - RF_DAILY) / r.std() * np.sqrt(252) if r.std() > 0 else float('nan')
    dd=((eq - eq.cummax()) / eq.cummax()).min()
    return {'Strategy':label, 'Total Ret (%)':round(tr*100,2), 'Ann. Ret (%)':round(ar*100,2),
            'Ann. Vol (%)':round(r.std()*np.sqrt(252)*100,2), 'Sharpe':round(sh,3),
            'Max DD (%)':round(dd*100,2), 'Calmar':round(ar/abs(dd),3) if dd else float('nan')}

eq_s=pd.Series(portfolio_OOS['equity'].values, index=portfolio_OOS.index)
m=compute_metrics(eq_s, 'Pairs Trading KC=F/BROS')
for k, v in m.items():print(f'  {k:<20s}:{v}')

  Strategy            :Pairs Trading KC=F/BROS
  Total Ret (%)       :9.15
  Ann. Ret (%)        :8.39
  Ann. Vol (%)        :44.42
  Sharpe              :0.305
  Max DD (%)          :-17.86
  Calmar              :0.47


## 9. Benchmark & Comparaison

In [9]:
bm_eq=CAPITAL * np.exp(log_prices[[ANCHOR, PAIR]].loc[test_idx].diff().dropna().mean(axis=1).cumsum())
bm_eq=pd.concat([pd.Series([float(CAPITAL)], index=[test_idx[0]]), bm_eq])
m_bm=compute_metrics(bm_eq, 'Buy & Hold KC=F+BROS')

keys=['Total Ret (%)', 'Ann. Ret (%)', 'Ann. Vol (%)', 'Sharpe', 'Max DD (%)', 'Calmar']
comparison=pd.DataFrame([{k:m[k] for k in keys}, {k:m_bm[k] for k in keys}],
                           index=['Pairs Trading KC=F/BROS', 'Buy & Hold KC=F+BROS'])
print(comparison.to_string())
comparison

                         Total Ret (%)  Ann. Ret (%)  Ann. Vol (%)  Sharpe  Max DD (%)  Calmar
Pairs Trading KC=F/BROS         9.1500        8.3900       44.4200  0.3050    -17.8600  0.4700
Buy & Hold KC=F+BROS          -14.1500      -13.0900       35.1000 -0.3410    -39.0400 -0.3350


,Total Ret (%),Ann. Ret (%),Ann. Vol (%),Sharpe,Max DD (%),Calmar
Pairs Trading KC=F/BROS,9.1500,8.3900,44.4200,0.3050,-17.8600,0.4700
Buy & Hold KC=F+BROS,-14.1500,-13.0900,35.1000,-0.3410,-39.0400,-0.3350


## 10. Performance

In [10]:
fig, axes=plt.subplots(2, 1, figsize=(12, 8))

ax=axes[0]
ax.plot(portfolio_OOS.index, portfolio_OOS['equity'], label='Pairs Trading KC=F/BROS', linewidth=1.8)
ax.plot(bm_eq.index, bm_eq.values, label='Buy & Hold KC=F+BROS', linewidth=1.5, linestyle='--')
ax.axhline(CAPITAL, color='grey', linestyle=':', linewidth=1)
ax.set_title('Portfolio OOS (2025-2026)'); ax.set_ylabel('Valeur ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _:f'${x:,.0f}'))
ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

ax=axes[1]
ax.plot(portfolio_OOS.index, portfolio_OOS['zscore'], color='steelblue', linewidth=1, label='Z-score')
ax.axhline( Z_THRESHOLD, color='red',   linestyle='--', linewidth=0.9, label=f'+{Z_THRESHOLD}')
ax.axhline(-Z_THRESHOLD, color='green', linestyle='--', linewidth=0.9, label=f'-{Z_THRESHOLD}')
ax.axhline(0, color='black', linewidth=0.7)
if len(trades):
    for _, t in trades.iterrows():
        if t['open_date'] in portfolio_OOS.index:
            ax.axvline(t['open_date'], color='green' if t['direction']=='LONG' else 'red', alpha=0.3, linewidth=0.8)
ax.set_title(f'Z-score OOS  (L={BEST_LB}j, KC=F/{PAIR})'); ax.set_ylabel('Z-score')
ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout(); plt.savefig('figures/fig_performance_g2.png', dpi=150, bbox_inches='tight'); plt.show()